In [ ]:
import simpy
import random
import numpy as np
import matplotlib.pyplot as plt
import heapq
from collections import defaultdict
import pandas as pd
import seaborn as sns

# Constants
WEEKS = 70
SIM_TIME = 10080 * WEEKS  # Simulation time in minutes
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

WEEKDAY_TRIAGENURSES = 3
triage_to_room_mean = 10.7
triage_to_room_std = 8.5
redirect_percentage = 0.70  # Percentage of ESI 4 and 5 patients redirected to urgent care
CLEANUP_TIME_MIN = 30
CLEANUP_TIME_MAX = 40


In [ ]:
# ED treatment times by ESI level
ed_treatment_times = {
    1: {'mean': 200.348718, 'std': 67.526061},
    2: {'mean': 379.571432, 'std': 353.184407},
    3: {'mean': 334.687680, 'std': 265.479295},
    4: {'mean': 173.150887, 'std': 168.620869},
    5: {'mean': 97.486852, 'std': 99.007531}
}

# Hospital treatment time (based on ESI level)
hospita_treatment_time = {
    1: {'mean': 1833.350000, 'std': 1673.970829},
    2: {'mean': 1705.736657, 'std': 1517.960759},
    3: {'mean': 1559.848562, 'std': 1419.780326},
    4: {'mean': 1143.018727, 'std': 1588.507923},
    5: {'mean': 3392.866667, 'std': 500}
}

# Early discharge flow times
ed_flow_times_early_disch = {
    1: {'mean': max(0, 1285.432051 - 180), 'std': 1553.596529},
    2: {'mean': max(0, 1129.072875 - 180), 'std': 1312.369788},
    3: {'mean': max(0, 657.596702 - 180), 'std': 917.206317},
    4: {'mean': max(0, 227.213370 - 180), 'std': 291.452199},
    5: {'mean': max(0, 146.559259 - 30), 'std': 230.024636}
}

# Admission probabilities based on triage level
admission_probabilities = {
    1: [0.615385, 0.384615],
    2: [0.447958, 0.552042],
    3: [0.194510, 0.805490],
    4: [0.016368, 0.983632],
    5: [0.003704, 0.996296]
}

# Final hospital dispositions
hospital_dispositions = {
    'Home Routine': 0.574273,
    'Home Healthcare IP Admit Related': 0.199124,
    'Skilled Nursing Facility': 0.083244,
    'Against Medical Advice': 0.008313,
    'Expired': 0.028239,
    'Rehab Facility': 0.012185,
    'Acute Care Facility': 0.005124,
    'Hospice Home': 0.020382,
    'Hospice in Place': 0.018790,
    'Jail/Prison': 0.001139,
    'Other': 0.049187
}

# Hospital treatment times by disposition
hospital_treatment_time_dispo = {
    'Home Routine': {'mean': 5621.703602, 'std': 4738.494399},
    'Home Healthcare IP Admit Related': {'mean': 9975.948882, 'std': 7927.979676},
    'Skilled Nursing Facility': {'mean': 13874.338441, 'std': 10350.488153},
    'Against Medical Advice': {'mean': 5793.606541, 'std': 5177.965441},
    'Expired': {'mean': 16737.622031, 'std': 14494.937959},
    'Rehab Facility': {'mean': 19541.864284, 'std': 15892.158489},
    'Acute Care Facility': {'mean': 12467.805796, 'std': 11785.769027},
    'Hospice Home': {'mean': 13837.502508, 'std': 10314.352608},
    'Hospice in Place': {'mean': 19528.281826, 'std': 15878.656889},
    'Jail/Prison': {'mean': 10298.931202, 'std': 8604.392339},
    'Other': {'mean': 19919.491146, 'std': 16310.964410}
}

# Triage definitions
triage_levels = [1, 2, 3, 4, 5]
triage_probabilities = [0.0003, 0.2577, 0.62, 0.116, 0.006]
triage_means = [5.698718, 4.697563, 4.397249, 3.954023, 3.539074]
triage_stds = [4.363339, 3.226411, 3.292795, 2.923584, 2.168832]


In [ ]:
# Function to run the simulation
def run_simulation(ed_occupy_bed_times, edip_stay_means, edip_stay_stds, urgent_care_redirect):
    # Initialize the simulation environment
    env = simpy.Environment()
    triage_nurses = simpy.Resource(env, capacity=WEEKDAY_TRIAGENURSES)
    total_cubicle_beds = 50
    hallway_beds_cap = 27
    surge_tent_beds_cap = 11
    total_edip_beds = 32
    occupancy_rate = 0.7
    hosp_beds_for_ed = 171
    pre_occ_hosp_beds = round(0.6 * hosp_beds_for_ed)
    initial_occupied_beds = int(total_cubicle_beds * occupancy_rate)
    
    cubicle_beds = simpy.PriorityResource(env, capacity=total_cubicle_beds)
    hallway_beds = simpy.PriorityResource(env, capacity=hallway_beds_cap)
    surge_tent_beds = simpy.PriorityResource(env, capacity=surge_tent_beds_cap)
    hospital_beds = simpy.PriorityResource(env, capacity=hosp_beds_for_ed)
    edip_bed = simpy.PriorityResource(env, capacity=total_edip_beds)

    census_data = {week: {day: 0 for day in range(7)} for week in range(WEEKS)}
    ed_edip_patients = set()

    arrivals_overall = {
        week: {
            day: [[] for _ in range(24)]
            for day in range(7)
        } for week in range(WEEKS)
    }

    bed_occupancy = {
        'Cubicle': {week: {day: [0]*24 for day in range(7)} for week in range(WEEKS)},
        'Hallway': {week: {day: [0]*24 for day in range(7)} for week in range(WEEKS)},
        'Surge': {week: {day: [0]*24 for day in range(7)} for week in range(WEEKS)}
    }

    total_patients = []
    arrivals_per_hour = {i: [[] for _ in range(24)] for i in range(7)}
    weekly_hospital_admissions = {week: {day: 0 for day in range(7)} for week in range(WEEKS)}

    # ... continuation in next chunk ...


In [ ]:
    # Inner class to manage patient data
    class Patient:
        def __init__(self, patient_id, arrival_time):
            self.patient_id = patient_id
            self.arrival_time = arrival_time
            self.triage_level = None
            self.triage_time = None
            self.ready_for_ed_bed_time = None
            self.wait_time_for_bed = None
            self.treatment_time = None
            self.bed_type = None
            self.admitted = None
            self.disposition = None
            self.hospital_disposition = None
            self.edip = None
            self.edip_start_time = None
            self.edip_end_time = None
            self.edip_duration = 0
            self.edip_bed_request = None
            self.hospital_treatment_time = None
            self.bed_request = None
            self.inpatient = False
            self.edip_bed_type = None
            self.is_ED_EDIP = False
            self.EDIP_Ward = False
            self.got_hosp_bed = False
            self.wait_start_time = 0
            self.ed_discharge_after_admission = 0

    # Function to pre-occupy ED cubicle beds
    def pre_occupy_beds(env, cubicle_beds, hallway_beds, initial_occupied):
        print(f"Starting with {initial_occupied} cubicle beds pre-occupied.")
        for i in range(initial_occupied):
            req = cubicle_beds.request()
            yield req
            triage_level = np.random.choice(triage_levels, p=triage_probabilities)
            mu, sigma = log_normal_params(
                ed_occupy_bed_times[triage_level]['mean'],
                ed_occupy_bed_times[triage_level]['std']
            )
            occupancy_time = np.random.lognormal(mu, sigma)
            print(f"Bed {i+1} occupied, will be occupied for {occupancy_time:.2f} minutes.")
            env.process(hold_bed(env, req, occupancy_time, cubicle_beds))

        assert cubicle_beds.count <= total_cubicle_beds, "Cubicle beds exceeded capacity after pre-occupying"
        assert hallway_beds.count <= hallway_beds_cap, "Hallway beds exceeded capacity after pre-occupying"

    # Function to pre-occupy hospital beds
    def pre_occupy_hospital_beds(env, initial_occupied):
        print(f"Starting with {initial_occupied} hospital beds pre-occupied.")
        dispositions = list(hospital_dispositions.keys())
        probabilities = list(hospital_dispositions.values())
        for i in range(initial_occupied):
            req = hospital_beds.request()
            yield req
            disposition = np.random.choice(dispositions, p=probabilities)
            mean = hospital_treatment_time_dispo[disposition]['mean']
            std = hospital_treatment_time_dispo[disposition]['std']
            mu, sigma = log_normal_params(mean, std)
            occupancy_time = np.random.lognormal(mu, sigma)
            print(f"Hospital bed {i+1} occupied, will be occupied for {occupancy_time:.2f} minutes.")
            env.process(hold_bed(env, req, occupancy_time, hospital_beds))

    # Helper: hold a bed for some time
    def hold_bed(env, req, time, beds):
        yield env.timeout(time)
        beds.release(req)

    # Log-normal parameter conversion
    def log_normal_params(mean, std):
        sigma = np.sqrt(np.log(1 + (std**2 / mean**2)))
        mu = np.log(mean**2 / np.sqrt(std**2 + mean**2))
        return mu, sigma


In [ ]:
    # Generate treatment time based on ESI level
    def generate_treatment_time(esi_level):
        mean = ed_occupy_bed_times[esi_level]['mean']
        std = ed_occupy_bed_times[esi_level]['std']
        mu, sigma = log_normal_params(mean, std)
        return np.random.lognormal(mu, sigma)

    # Generate hospital treatment time based on disposition
    def generate_hospital_treatment_time_dispo_based(disposition):
        mean = hospital_treatment_time_dispo[disposition]['mean']
        std = hospital_treatment_time_dispo[disposition]['std']
        mu, sigma = log_normal_params(mean, std)
        return np.random.lognormal(mu, sigma)

    # Assign triage level and time
    def assign_triage_level(patient):
        patient.triage_level = np.random.choice(triage_levels, p=triage_probabilities)
        mean = triage_means[patient.triage_level - 1]
        std = triage_stds[patient.triage_level - 1]
        mu = np.log(mean**2 / np.sqrt(mean**2 + std**2))
        sigma = np.sqrt(np.log(1 + (std**2 / mean**2)))
        patient.triage_time = np.random.lognormal(mean=mu, sigma=sigma)
        print(f"Patient {patient.patient_id} assigned to triage level {patient.triage_level} with triage time {patient.triage_time:.2f} minutes.")

    # Generate hourly patient arrivals
    def generate_hourly_data(daily_avg, sd, hourly_distribution):
        arrival_rates = []
        for avg in daily_avg:
            daily_total = round(np.random.normal(avg, sd))
            hourly_data = [daily_total * perc for perc in hourly_distribution]
            hourly_data = [max(1, round(val)) for val in hourly_data]
            difference = daily_total - sum(hourly_data)
            while difference != 0:
                for i in range(len(hourly_data)):
                    if difference == 0:
                        break
                    if difference > 0 and hourly_data[i] < daily_total * hourly_distribution[i]:
                        hourly_data[i] += 1
                        difference -= 1
                    elif difference < 0 and hourly_data[i] > 1:
                        hourly_data[i] -= 1
                        difference += 1
            arrival_rates.append(hourly_data)
        return arrival_rates

    # Simulate patient arrival
    def patient_arrival(env, daily_avg, sd, hourly_distribution):
        patient_id = 0
        for week in range(WEEKS):
            weekly_arrival_rates = generate_hourly_data(daily_avg, sd, hourly_distribution)
            for day in range(7):
                for hour in range(24):
                    arrival_rate = weekly_arrival_rates[day][hour]
                    interarrival_time = 60 / arrival_rate if arrival_rate > 0 else 60
                    for _ in range(round(arrival_rate)):
                        yield env.timeout(interarrival_time)
                        patient_id += 1
                        patient = Patient(patient_id, env.now)
                        env.process(handle_triage(env, patient, week, day, hour))
                        arrivals_per_hour[day][hour].append(patient)
                        arrivals_overall[week][day][hour].append(patient)
                        total_patients.append(patient)
                        print(f"Patient {patient.patient_id} arrives at hour {hour} on day {day}.")



In [ ]:
    # Handle triage process
    def handle_triage(env, patient, week, day, hour):
        assign_triage_level(patient)
        print(f"Patient {patient.patient_id} trying to request a nurse at {env.now}.")
        print(f"Queue length before requesting nurse: {len(triage_nurses.queue)}")

        with triage_nurses.request() as req:
            yield req
            print(f"Patient {patient.patient_id} nurse assigned at {env.now}.")
            start_wait = env.now

            # Perform triage
            yield env.timeout(patient.triage_time)
            triage_time = env.now - start_wait
            print(f"Patient {patient.patient_id} completed triage at {env.now}, waited {triage_time:.2f} mins, triage time: {patient.triage_time:.2f} minutes.")

            # Simulate cleanup time
            cleanup_time = np.random.uniform(10, 20)
            yield env.timeout(cleanup_time)
            print(f"Cleanup completed, nurse available again at {env.now}, cleanup took {cleanup_time:.2f} minutes.")
            print(f"Queue length after completing triage and cleanup: {len(triage_nurses.queue)}")

        patient.ready_for_bed_time = env.now
        post_triage_delay = np.random.uniform(10, 20)
        yield env.timeout(post_triage_delay)
        print(f"Post-triage delay completed at {env.now}, delay was {post_triage_delay:.2f} minutes.")

        if urgent_care_redirect and patient.triage_level in [4, 5] and np.random.rand() < redirect_percentage:
            env.process(handle_urgent_care(env, patient))
        else:
            env.process(handle_patient_to_room(env, patient, week, day, hour))

    # Handle patients redirected to urgent care
    def handle_urgent_care(env, patient):
        urgent_care_time = np.random.uniform(60, 240)
        yield env.timeout(urgent_care_time)
        print(f"Patient {patient.patient_id} redirected to urgent care and processed in {urgent_care_time:.2f} minutes.")


In [ ]:
    # Handle patient room assignment
    def handle_patient_to_room(env, patient, week, day, hour):
        if cubicle_beds.count < cubicle_beds.capacity:
            request = cubicle_beds.request(priority=patient.triage_level)
            bed_type = 'Cubicle'
        elif hallway_beds.count < hallway_beds.capacity:
            request = hallway_beds.request(priority=patient.triage_level)
            bed_type = 'Hallway'
        elif surge_tent_beds.count < surge_tent_beds.capacity:
            request = surge_tent_beds.request(priority=patient.triage_level)
            bed_type = 'Surge'
        else:
            cubicle_request = cubicle_beds.request(priority=patient.triage_level)
            hallway_request = hallway_beds.request(priority=patient.triage_level)
            surge_request = surge_tent_beds.request(priority=patient.triage_level)
            results = yield cubicle_request | hallway_request | surge_request

            if cubicle_request in results:
                request = cubicle_request
                hallway_request.cancel()
                surge_request.cancel()
                bed_type = 'Cubicle'
            elif hallway_request in results:
                request = hallway_request
                cubicle_request.cancel()
                surge_request.cancel()
                bed_type = 'Hallway'
            else:
                request = surge_request
                cubicle_request.cancel()
                hallway_request.cancel()
                bed_type = 'Surge'

        yield request
        patient.bed_request = request
        patient.bed_type = bed_type
        actual_wait_time = env.now - patient.ready_for_bed_time
        patient.wait_time_for_bed = actual_wait_time

        print(f"Patient {patient.patient_id} assigned to a {bed_type} bed after waiting {actual_wait_time:.2f} minutes.")
        assert cubicle_beds.count <= total_cubicle_beds, f"Cubicle beds exceeded capacity after assigning bed: {cubicle_beds.count}"
        assert hallway_beds.count <= hallway_beds_cap, f"Hallway beds exceeded capacity after assigning bed: {hallway_beds.count}"

        # Simulate time spent in treatment
        treatment_time = generate_treatment_time(patient.triage_level)
        patient.treatment_time = treatment_time
        print(f"Patient {patient.patient_id} will be in bed for {treatment_time:.2f}.")
        yield env.timeout(treatment_time)

        # Handle disposition after treatment
        assign_admission_and_disposition(patient, request, week, day, hour)


In [ ]:
    # Decide if patient is admitted or discharged
    def assign_admission_and_disposition(patient, request, week, day, hour):
        patient.admitted = np.random.choice([True, False], p=admission_probabilities[patient.triage_level])

        if patient.admitted:
            print(f'Patient {patient.patient_id} will be admitted')
            env.process(handle_upstairs_bed(env, patient, week, day, hour))
        else:
            print(f'Patient {patient.patient_id} will be discharged')
            env.process(release_bed(patient, week, day, hour))

    # Assign hospital disposition
    def assign_hospital_disposition(patient):
        dispositions = list(hospital_dispositions.keys())
        probabilities = list(hospital_dispositions.values())
        patient.hospital_disposition = np.random.choice(dispositions, p=probabilities)

    # Release ED bed and simulate cleanup
    def release_bed(patient, week, day, hour):
        print(f"Patient {patient.patient_id} bed released at {env.now:.2f}")
        cleanup_time = np.random.uniform(CLEANUP_TIME_MIN, CLEANUP_TIME_MAX)
        print(f"Bed will be cleaned now for {cleanup_time:.2f} mins")
        yield env.timeout(cleanup_time)

        if patient.bed_type == 'Cubicle':
            cubicle_beds.release(patient.bed_request)
        elif patient.bed_type == 'Hallway':
            hallway_beds.release(patient.bed_request)
        else:
            surge_tent_beds.release(patient.bed_request)

        queue_length = len(cubicle_beds.queue) if patient.bed_type == 'Cubicle' else (
            len(hallway_beds.queue) if patient.bed_type == 'Hallway' else len(surge_tent_beds.queue)
        )
        print(f"Queue length after Patient {patient.patient_id} leaves the {patient.bed_type} bed: {queue_length}")


In [ ]:
    def handle_upstairs_bed(env, patient, week, day, hour):
        print(f"Handling upstairs bed for Patient {patient.patient_id} at {env.now}")
        
        assign_hospital_disposition(patient)
        hosp_treatment_time = generate_hospital_treatment_time_dispo_based(patient.hospital_disposition)
        patient.hospital_treatment_time = hosp_treatment_time
        patient.wait_start_time = env.now

        env.process(monitor_wait_time_for_exiting_ED(env, patient))
        print(f"{patient.patient_id} treatment time: {hosp_treatment_time} minutes)")

        print(f"EDIP bed availability: {edip_bed.capacity - edip_bed.count}, Hospital bed availability: {hospital_beds.capacity - hospital_beds.count}")
        req_hospital_bed = hospital_beds.request()
        result = yield req_hospital_bed | env.timeout(0)

        if req_hospital_bed in result:
            env.process(release_bed(patient, week, day, hour))
            current_week, current_day = calculate_time_units(env.now)
            weekly_hospital_admissions[current_week][current_day] += 1
            if patient.is_ED_EDIP:
                ed_edip_patients.discard(patient.patient_id)
            patient.ed_discharge_after_admission = env.now - patient.wait_start_time
            patient.got_hosp_bed = True
            print(f"Hospital bed assigned immediately to Patient {patient.patient_id} at {env.now}")
            yield env.timeout(hosp_treatment_time)
            print(f"Patient {patient.patient_id} leaves the Hospital bed at {env.now}")
            hospital_beds.release(req_hospital_bed)

        else:
            print(f"Hospital bed not immediately available for Patient {patient.patient_id}, requesting EDIP bed at {env.now}")
            req_edip_bed = edip_bed.request()
            result = yield env.any_of([env.timeout(hosp_treatment_time), req_hospital_bed, req_edip_bed])
            remaining_treatment_time = hosp_treatment_time - (env.now - patient.wait_start_time)

            if env.timeout(hosp_treatment_time) in result:
                if patient.is_ED_EDIP:
                    ed_edip_patients.discard(patient.patient_id)
                patient.ED_EDIP = True
                patient.ed_discharge_after_admission = env.now - patient.wait_start_time
                print(f"Patient {patient.patient_id} spent entire treatment time waiting in the ED and is discharged at {env.now}.")
                return

            if req_hospital_bed in result:
                env.process(release_bed(patient, week, day, hour))
                if patient.is_ED_EDIP:
                    ed_edip_patients.discard(patient.patient_id)
                current_week, current_day = calculate_time_units(env.now)
                weekly_hospital_admissions[current_week][current_day] += 1
                patient.ed_discharge_after_admission = env.now - patient.wait_start_time
                patient.got_hosp_bed = True
                edip_bed.release(req_edip_bed)
                yield env.timeout(remaining_treatment_time)
                hospital_beds.release(req_hospital_bed)
                print(f"Patient {patient.patient_id} leaves the Hospital bed at {env.now}")

            elif req_edip_bed in result:
                patient.EDIP_Ward = True
                patient.edip_start = env.now
                env.process(release_bed(patient, week, day, hour))
                if patient.is_ED_EDIP:
                    ed_edip_patients.discard(patient.patient_id)
                patient.ed_discharge_after_admission = env.now - patient.wait_start_time
                print(f"EDIP bed assigned to Patient {patient.patient_id} at {env.now}")

                treatment_done = env.timeout(remaining_treatment_time)
                result = yield env.any_of([treatment_done, req_hospital_bed])

                if treatment_done in result:
                    patient.edip_end = env.now
                    patient.edip_duration = patient.edip_end - patient.edip_start
                    edip_bed.release(req_edip_bed)
                    print(f"Patient {patient.patient_id} completed treatment in EDIP bed at {env.now}")

                elif req_hospital_bed in result:
                    patient.got_hosp_bed = True
                    patient.edip_end = env.now
                    patient.edip_duration = patient.edip_end - patient.edip_start
                    current_week, current_day = calculate_time_units(env.now)
                    weekly_hospital_admissions[current_week][current_day] += 1
                    edip_bed.release(req_edip_bed)
                    yield env.timeout(remaining_treatment_time)
                    hospital_beds.release(req_hospital_bed)
                    print(f"Patient {patient.patient_id} leaves the Hospital bed at {env.now}")


In [ ]:
    # Monitor if patient becomes ED_EDIP due to long wait
    def monitor_wait_time_for_exiting_ED(env, patient, max_wait_time=120):
        yield env.timeout(max_wait_time)
        if not patient.got_hosp_bed and not patient.EDIP_Ward:
            patient.is_ED_EDIP = True
            ed_edip_patients.add(patient.patient_id)
            print(f"Patient {patient.patient_id} has been waiting for more than 2 hours and is marked as ED_EDIP at {env.now}.")

    # Calculate current week and day
    def calculate_time_units(sim_time):
        week = int(sim_time // (7 * 24 * 60))  # 1 week = 7d * 24h * 60min
        day = int((sim_time % (7 * 24 * 60)) // (24 * 60))
        return week, day

    # EDIP beds open/close by hour of day
    def manage_edip_beds(env, edip_bed_resource, total_beds):
        closed_bed_requests = []
        while True:
            current_hour = (env.now % (24 * 60)) // 60
            if 6 <= current_hour < 18:
                available_beds_to_close = total_beds - edip_bed_resource.count
                beds_to_close = min(available_beds_to_close, 9 - len(closed_bed_requests))
                if beds_to_close > 0:
                    print(f"Closing {beds_to_close} EDIP beds at {env.now // 60} hours.")
                    for _ in range(beds_to_close):
                        req = edip_bed_resource.request(priority=float('-inf'))
                        yield req
                        closed_bed_requests.append(req)
                else:
                    print("Beds occupied, can't close")
            else:
                if closed_bed_requests:
                    print(f"Reopening {len(closed_bed_requests)} EDIP beds at {env.now // 60} hours.")
                    while closed_bed_requests:
                        req = closed_bed_requests.pop(0)
                        edip_bed_resource.release(req)
            yield env.timeout(60)


In [ ]:
    # Daily 7 a.m. census count
    def census_at_7am(env):
        while True:
            now = env.now
            current_hour = (now % (24 * 60)) // 60
            time_until_7am = ((7 - current_hour) % 24) * 60
            yield env.timeout(time_until_7am)

            ed_edip_count = len(ed_edip_patients) + edip_bed.count
            print(f"Census at 7 a.m.: {ed_edip_count} patients still in ED marked as EDIP at time {env.now}")
            print(f"Breakdown: {len(ed_edip_patients)} in ED EDIP, {edip_bed.count} in EDIP beds at {env.now}")
            current_week, current_day = calculate_time_units(env.now)
            census_data[current_week][current_day] = ed_edip_count

            yield env.timeout(24 * 60)

    # Hourly bed occupancy tracking
    def update_bed_occupancy(env, bed_occupancy):
        while True:
            current_time = int(env.now)
            current_hour = (current_time // 60) % 24
            current_day = (current_time // (24 * 60)) % 7
            current_week = current_time // (7 * 24 * 60)

            bed_occupancy['Cubicle'][current_week][current_day][current_hour] = cubicle_beds.count
            bed_occupancy['Hallway'][current_week][current_day][current_hour] = hallway_beds.count
            bed_occupancy['Surge'][current_week][current_day][current_hour] = surge_tent_beds.count

            print(f"Hour {current_hour} on Day {current_day}, Week {current_week}:")
            print(f"Cubicles: {cubicle_beds.count}, Hallways: {hallway_beds.count}, Surge: {surge_tent_beds.count}")

            yield env.timeout(60)

    # Patient arrival configuration
    daily_avg = [123, 123, 118, 117, 114, 97, 96]
    sd = 11
    hourly_distribution = [
        0.021, 0.017, 0.013, 0.011, 0.012, 0.013, 0.015, 0.022, 0.032, 0.055,
        0.069, 0.074, 0.072, 0.068, 0.068, 0.067, 0.064, 0.062, 0.055, 0.051,
        0.045, 0.038, 0.031, 0.025
    ]

    # Initialization wrapper
    def initialize_simulation(env, cubicle_beds, hallway_beds, initial_occupied, daily_avg, sd, hourly_distribution):
        yield env.process(pre_occupy_beds(env, cubicle_beds, hallway_beds, initial_occupied))
        yield env.process(pre_occupy_hospital_beds(env, pre_occ_hosp_beds))
        env.process(update_bed_occupancy(env, bed_occupancy))
        env.process(census_at_7am(env))
        env.process(manage_edip_beds(env, edip_bed, total_edip_beds))
        env.process(patient_arrival(env, daily_avg, sd, hourly_distribution))

    # Start simulation
    env.process(initialize_simulation(env, cubicle_beds, hallway_beds, initial_occupied_beds, daily_avg, sd, hourly_distribution))
    env.run(until=SIM_TIME)

    return total_patients, bed_occupancy, arrivals_overall, weekly_hospital_admissions, census_data


In [ ]:
def analyze_patient_data(total_patients):
    # Prepare the data into a DataFrame
    data = {
        'Triage Time': [patient.triage_time for patient in total_patients],
        'Wait Time for Bed': [patient.wait_time_for_bed for patient in total_patients],
        'Treatment Time': [patient.treatment_time for patient in total_patients],
        'Arrival Time': [patient.arrival_time for patient in total_patients],
        'EDIP and IP': [
            patient.edip_duration + patient.ed_discharge_after_admission
            if (patient.is_ED_EDIP or patient.EDIP_Ward) and patient.got_hosp_bed else None
            for patient in total_patients
        ],
        'Completely EDIP': [
            patient.edip_duration + patient.ed_discharge_after_admission
            if not patient.got_hosp_bed and (patient.is_ED_EDIP or patient.EDIP_Ward) else None
            for patient in total_patients
        ]
    }

    df = pd.DataFrame(data)
    df['combined EDIP'] = df['EDIP and IP'] + df['Completely EDIP']

    # Summary statistics
    edip_ward_patients = sum(1 for p in total_patients if p.EDIP_Ward)
    print(f"Total EDIP (Ward) Patients: {edip_ward_patients}")

    inpatient_count = sum(1 for p in total_patients if p.got_hosp_bed)
    print(f"Number of Inpatients: {inpatient_count}")

    edip_then_ip = sum(1 for p in total_patients if (p.is_ED_EDIP or p.EDIP_Ward) and p.got_hosp_bed)
    print(f"Number of EDIP then Inpatients: {edip_then_ip}")

    edip_only = sum(1 for p in total_patients if not p.got_hosp_bed and (p.is_ED_EDIP or p.EDIP_Ward))
    print(f"Number EDIP only (ED and EDIP Ward): {edip_only}")

    ed_edip = sum(1 for p in total_patients if not p.got_hosp_bed and p.is_ED_EDIP)
    print(f"Number of patients who got discharged from the ED (didn't get a bed anywhere): {ed_edip}")

    in_ed_for_long = sum(1 for p in total_patients if p.is_ED_EDIP)
    print(f"Number of patients who spent > 2 hours in the ED (after disposition admitted): {in_ed_for_long}")

    # Time between arrivals
    time_differences = [
        total_patients[i].arrival_time - total_patients[i-1].arrival_time
        for i in range(1, len(total_patients))
    ]
    print(f"Total number of patients: {len(total_patients)}")
    if time_differences:
        print(f"Average time difference between patient arrivals: {np.mean(time_differences)}")
        print(f"Minimum time difference: {min(time_differences)}")
        print(f"Maximum time difference: {max(time_differences)}")
    else:
        print("Not enough data to calculate time differences between patient arrivals.")

    # Wait time stats
    max_wait_time_for_bed = df['Wait Time for Bed'].max()
    print(f"Maximum wait time for a bed: {max_wait_time_for_bed}")
    mean_wait_time_for_bed = df['Wait Time for Bed'].mean()
    print(f"Mean wait time for a bed: {mean_wait_time_for_bed}")

    return df, edip_ward_patients, time_differences


In [ ]:
def create_visuals(df, arrivals_overall):
    # Average hourly arrivals per day
    total_arrivals_per_hour = {day: [0] * 24 for day in range(7)}
    week_counts = {day: 0 for day in range(7)}

    for week, week_data in arrivals_overall.items():
        for day, hours in week_data.items():
            for hour in range(24):
                total_arrivals_per_hour[day][hour] += len(hours[hour])
            week_counts[day] += 1

    avg_arrivals_per_hour = {
        day: [total / week_counts[day] for total in hours]
        for day, hours in total_arrivals_per_hour.items()
    }

    # Plot: Hourly arrivals
    days_of_week = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    plt.figure(figsize=(12, 8))
    for day in range(7):
        plt.plot(avg_arrivals_per_hour[day], label=days_of_week[day])
    plt.xlabel('Hour of the Day')
    plt.ylabel('Average Number of Arrivals')
    plt.title('Average Arrivals per Hour for Each Day')
    plt.legend(title='Day of the Week')
    plt.grid(True)
    plt.show()

    # Histogram: EDIP then IP Length of Stay
    plt.figure(figsize=(10, 6))
    plt.hist(df['EDIP and IP'].dropna(), bins=100, edgecolor='k', alpha=0.7)
    plt.xlabel('EDIP LOS')
    plt.ylabel('Frequency')
    plt.title('LOS in EDIP/ED for people who were EDIP and then got a bed upstairs')
    plt.grid(True)
    plt.show()

    # Histogram: Completely EDIP patients
    plt.figure(figsize=(10, 6))
    plt.hist(df['Completely EDIP'].dropna(), bins=30, edgecolor='k', alpha=0.7)
    plt.xlabel('Length of Stay in ED (after disposition)')
    plt.ylabel('Frequency')
    plt.title('LOS in EDIP/ED for people who were EDIP and got discharged from there')
    plt.grid(True)
    plt.show()

    # Histogram: Triage Time distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Triage Time'], kde=True, color="skyblue", bins=30)
    plt.title('Distribution of Triage Time')
    plt.xlabel('Triage Time (minutes)')
    plt.ylabel('Frequency')
    plt.show()

    # Histogram: Wait Time for Bed (filtered)
    threshold = df['Wait Time for Bed'].quantile(0.90)
    filtered_df = df[df['Wait Time for Bed'] <= threshold]

    plt.figure(figsize=(10, 6))
    sns.histplot(filtered_df['Wait Time for Bed'], kde=True, color="olive", bins=30)
    plt.title('Distribution of Wait Time for Bed')
    plt.xlabel('Wait Time for Bed (minutes)')
    plt.ylabel('Frequency')
    plt.show()

    # Histogram: Log-transformed Treatment Time
    df['Log ED LOS'] = np.log1p(df['Treatment Time'])
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Log ED LOS'], kde=True, color="gold", bins=30)
    plt.title('Distribution of ED LOS (only treatment time)')
    plt.xlabel('ED LOS in log (minutes)')
    plt.ylabel('Frequency')
    plt.show()

    # Histogram: Total ED Length of Stay (filtered)
    df['Total ED Length'] = df['Triage Time'] + df['Wait Time for Bed'] + df['Treatment Time']
    threshold_forED = df['Total ED Length'].quantile(0.95)
    filtered_df_ED = df[df['Total ED Length'] <= threshold_forED]

    plt.figure(figsize=(10, 6))
    sns.histplot(filtered_df_ED['Total ED Length'], kde=True, color="olive", bins=30)
    plt.title('Distribution of total ED Length of Stay')
    plt.xlabel('Time spent in ED')
    plt.ylabel('Frequency')
    plt.show()

    # Histogram: Log-transformed Wait Time for Bed
    df['Log Wait Time for Bed'] = np.log1p(df['Wait Time for Bed'])
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Log Wait Time for Bed'], kde=True, color="olive", bins=30)
    plt.title('Log-Transformed Distribution of Wait Time for Bed')
    plt.xlabel('Log Wait Time for Bed (log1p(minutes))')
    plt.ylabel('Frequency')
    plt.show()


In [ ]:
def visualize_patient_data_for_EDIP(data):
    records = []
    for ward, weeks in data.items():
        for week, days in weeks.items():
            for day, hours in days.items():
                for hour, count in enumerate(hours):
                    records.append([ward, week, day, hour, count])
    
    df = pd.DataFrame(records, columns=['Ward', 'Week', 'Day', 'Hour', 'Count'])
    
    # Aggregate by week
    weekly_data = df.groupby(['Ward', 'Week'])['Count'].sum().unstack(level=0)
    
    # Plot each ward individually
    for ward in data.keys():
        plt.figure(figsize=(12, 6))
        plt.plot(weekly_data.index, weekly_data[ward], marker='o', label=ward)
        plt.title(f'Number of Patients in {ward} Over the Course of a Year')
        plt.xlabel('Week')
        plt.ylabel('Number of Patients')
        plt.legend()
        plt.grid(True)
        plt.show()
    
    # Combined plot
    plt.figure(figsize=(12, 6))
    for ward in data.keys():
        plt.plot(weekly_data.index, weekly_data[ward], marker='o', label=ward)
    plt.title('Number of Patients in Different Wards Over the Course of a Year')
    plt.xlabel('Week')
    plt.ylabel('Number of Patients')
    plt.legend()
    plt.grid(True)
    plt.show()
